# Aedes-AI GRU Model with Mean Temperature as Input


This code trains and tests a GRU model that estimates Aedes aegypti abundance from the following time series.

* Daily mean temperature (C)
* Daily precipitation (cm)
* Daily relative humidity

The GRU is trained to reproduce the results of MoLS.

In [20]:
import sys, importlib, json, os
sys.path.append('../')

import pandas as pd
import numpy as np
import tensorflow as tf

import matplotlib.pyplot as plt

import utils.finetune_training as finetune
import utils.forecast as forecast
import utils.predictions as predictions

# Opening config file
f = open('../fpaths_config.json')
paths = json.load(f)

model_fils_path = paths["model_files"]
mols_path = paths["raw_mols"]

### First for San Juan

In [11]:
data_fil = '../data/raw_mols_predictions/San_Juan_MoLS.csv'
data = pd.read_csv(data_fil)

data = data.iloc[180:-180,:].reset_index(drop=True)

data.to_csv('../utils/model_files/finetune_sj.csv', index=False)



In [25]:
importlib.reload(finetune)

model_file = '{}/gru_avg_temp.h5'.format(model_fils_path)
base_model = tf.keras.models.load_model(model_file, custom_objects = {'r2_keras': predictions.r2_keras})
# Opening config file
f = open('{}/gru_avg_temp_config.json'.format(model_fils_path))
config = json.load(f)

scaler = pd.read_pickle('{}/avg_scaler.pkl'.format(model_fils_path))

#Load the data
data = pd.read_csv('{}/finetune_sj.csv'.format(model_fils_path))

##Train on 2000-2015 San Juan, val on 2016 San Juan
val = data[data.Year==2016]
val.reset_index(drop=True, inplace=True)

train = data[data.Year<2016]
train.reset_index(drop=True, inplace=True)

#val = val.loc[0:365, :]
#val.reset_index(drop=True, inplace=True)
X_val, y_val, locs_val = finetune.format_sj_mols(val, scaler)

#train = data[data.Location.str.contains('San_Juan')]

yrs = ['all', 1, 3, 5, 10]
for yr in yrs:
    finetuned_model = tf.keras.models.clone_model(base_model)
    finetuned_model.set_weights(base_model.get_weights())

    for layer in finetuned_model.layers[:-2]:
        layer.trainable = False

    subset = train.copy(deep=True)
    if not yr == 'all':
        subset = subset.iloc[-(yr*365):-1,:]
    
    subset.reset_index(drop=True, inplace=True)
        
    #Format training and validation data
    X_train, y_train, locs_train = finetune.format_sj_mols(subset, scaler)
    print(yr, subset.shape, X_train.shape)
    permutation = np.random.permutation(len(X_train))
    X_train = X_train[permutation, :, :]
    y_train = y_train[permutation, :]
    locs_train = locs_train[permutation, :]

    #Fine-tune model by setting a lower learning rate
    finetuned_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.00005), loss='mse', metrics=[predictions.r2_keras])

    history_fine_tune = finetuned_model.fit(X_train, y_train, validation_data = (X_val, y_val), batch_size=64, epochs=200,
                callbacks = [tf.keras.callbacks.TensorBoard(), tf.keras.callbacks.EarlyStopping(patience = 30, restore_best_weights = True)])
    finetuned_model.save('{}/finetuned_models/finetuned_sj_{}.h5'.format(model_fils_path, yr), save_format = 'h5')

c:\Users\Adrienne\Documents\Projects\Aedes\Aedes-AI_Forecasting\notebooks\..\utils\finetune_training.py:90: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Datetime'] = pd.to_datetime(data[['Year', 'Month', 'Day']])


all (5638, 9) (5458, 90, 3)
Epoch 1/200
86/86 [==============================] - 6s 42ms/step - loss: 0.0021 - r2_keras: 0.8979 - val_loss: 0.0149 - val_r2_keras: -2.5993
Epoch 2/200
86/86 [==============================] - 3s 36ms/step - loss: 0.0016 - r2_keras: 0.9230 - val_loss: 0.0125 - val_r2_keras: -2.1690
Epoch 3/200
86/86 [==============================] - 3s 36ms/step - loss: 0.0014 - r2_keras: 0.9309 - val_loss: 0.0125 - val_r2_keras: -2.2227
Epoch 4/200
86/86 [==============================] - 3s 37ms/step - loss: 0.0013 - r2_keras: 0.9361 - val_loss: 0.0121 - val_r2_keras: -2.1439
Epoch 5/200
86/86 [==============================] - 3s 37ms/step - loss: 0.0012 - r2_keras: 0.9388 - val_loss: 0.0106 - val_r2_keras: -1.8390
Epoch 6/200
86/86 [==============================] - 3s 36ms/step - loss: 0.0012 - r2_keras: 0.9425 - val_loss: 0.0110 - val_r2_keras: -1.9420
Epoch 7/200
86/86 [==============================] - 3s 36ms/step - loss: 0.0011 - r2_keras: 0.9455 - val_loss: 0.